# Step 07: Model Comparison & Selection Benchmark

## Overview
This notebook benchmarks three classification algorithms on identical train/test splits and preprocessing pipelines:
1. **Logistic Regression** (`class_weight='balanced'`)
2. **Random Forest** (`class_weight='balanced'`)
3. **XGBoost** (`scale_pos_weight` tuned for class imbalance)

### Model Selection Criteria (Ground Rule 4):
In employee attrition prediction, **False Negatives (missing a real flight risk)** are vastly more expensive than False Positives (a false alarm HR check-in). We select the winning model based on **Recall**, **F1-Score**, and minimizing **False Negatives**, rather than raw accuracy.


In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_engineered.csv"))

drop_cols = ['EmployeeNumber', 'Employee ID', 'Attrition', 'Target_Attrition']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols]
y = df['Target_Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)


---
## 1. Train & Benchmark 3 Classifiers


In [2]:
scale_pos_wt = (len(y_train) - y_train.sum()) / float(y_train.sum())

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(n_estimators=150, scale_pos_weight=scale_pos_wt, random_state=42, eval_metric='logloss')
}

results = []
pipelines = {}

for name, clf in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    cm = confusion_matrix(y_test, y_pred)
    fn = cm[1, 0] # False Negatives
    fp = cm[0, 1] # False Positives
    
    results.append({
        "Model": name,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1,
        "ROC-AUC": auc,
        "False Negatives (Missed)": fn,
        "False Positives (False Alarm)": fp
    })

results_df = pd.DataFrame(results).sort_values(by='Recall', ascending=False)
print("=== Model Comparison Benchmark Table ===")
print(results_df.to_string(index=False))


=== Model Comparison Benchmark Table ===
              Model  Precision   Recall  F1-Score  ROC-AUC  False Negatives (Missed)  False Positives (False Alarm)
Logistic Regression   0.369048 0.659574  0.473282 0.804893                        16                             53
            XGBoost   0.583333 0.297872  0.394366 0.757774                        33                             10
      Random Forest   0.600000 0.063830  0.115385 0.768671                        44                              2


---
## 2. Winning Model Selection & Persistence


In [3]:
# Select model with highest recall (lowest False Negatives) and top ROC-AUC
winner_name = results_df.iloc[0]['Model']
winning_pipeline = pipelines[winner_name]

out_model_path = os.path.join(MODELS_DIR, "attrition_pipeline.joblib")
joblib.dump(winning_pipeline, out_model_path)

print(f"🏆 WINNER SELECTED: {winner_name}")
print(f"✔ Pipeline saved to: {out_model_path}")


🏆 WINNER SELECTED: Logistic Regression
✔ Pipeline saved to: ..\models\attrition_pipeline.joblib
